In [1]:
import matplotlib.pyplot as plt
from datetime import datetime, timedelta
import pandas as pd
import numpy as np
import pytz
import re
import matplotlib.dates as mdates


In [2]:
# # populate total_scount

def calculate_total_scount(df):
    #print("calculate_total_scount")

    # Initialize the total_scount column with None or a default value
    if "total_scount" not in df.columns:
        df["total_scount"] = 0

    # Iterate over the rows
    for i in range(len(df)):
        current_ta = df.at[i, "wlan.ta_int"]
        current_scount = df.at[i, "wlan.qbss.scount"]

        # Look for a previous record where wlan.ta_int is one higher or lower
        j = i - 1  # Start with the immediately preceding row
        while j >= 0:
            prev_ta = df.at[j, "wlan.ta_int"]
            if abs(prev_ta - current_ta) == 1:
                # Found a previous record meeting the condition
                prev_scount = df.at[j, "wlan.qbss.scount"]
                df.at[i, "total_scount"] = current_scount + prev_scount
                break
            j -= 1  # Go to the next previous row

        # If no valid previous record was found, assign total_scount based on the next record's total_scount
        # this is broken
        if j < 0:
            k = i + 1  # Start with the next row
            while k < len(df):
                next_ta = df.at[k, "wlan.ta_int"]
                if next_ta == current_ta:
                    # Found the next record with the same wlan.ta_int
                    next_total_scount = df.at[k, "total_scount"]
                    df.at[i, "total_scount"] = next_total_scount
                    break
                k += 1
            
            # If no next record exists, keep total_scount as current_scount
            if k == len(df):
                df.at[i, "total_scount"] = current_scount

    # Copy the value of total_scount from the second record to the first record, improves min values and graphs
    # this needs to be done more elegantly later in case beacons are missing
    # calculate_total_scount was edited to correctly reflect first few rows
    #df.loc[0, "total_scount"] = df.loc[1, "total_scount"]
    #df.loc[2, "total_scount"] = df.loc[3, "total_scount"]
    
    # Ensure total_scount is of type integer
    df["total_scount"] = df["total_scount"].astype(int)

    return df

In [3]:
#  Define the function to generate the plot
#inputs are dataframes filtered to the radio, and further filtered for each SSID
def plot_channel_stations(radio_df, eduroam_df, byu_wifi_df, AP_name, start_time, end_time):
    #print("plot_channel_stations")
    plt.figure(figsize=(14, 7))  # Example size: 12 inches wide, 6 inches tall
    # Add a horizontal line at value 70 with 20% opacity
    plt.axhline(y=70, color='red', linestyle='-', alpha=0.2, zorder=-1)

    # Access the first record and get the value of 'radio_number'
    plot_radio_number = radio_df.iloc[0]["radio_number"]
    
    # Plot wlan.qbss.cu normalized as percentages
    plt.plot(radio_df["aruba_erm.time"], (radio_df["wlan.qbss.cu"] / 255) * 100, label="Channel Utilization (%)")

    # Fill the area below the line
    plt.fill_between(radio_df["aruba_erm.time"], (radio_df["wlan.qbss.cu"] / 255) * 100)


    # Plot total_scount
    plt.plot(radio_df["aruba_erm.time"], radio_df["total_scount"], label="Total Station Count")

    # Plot total_scount for "eduroam"
    #plt.plot(eduroam_df["aruba_erm.time"], eduroam_df["wlan.qbss.scount"], label="eduroam Station Count")

    # Plot total_scount for "BYU-WiFi"
    #plt.plot(byu_wifi_df["aruba_erm.time"], byu_wifi_df["wlan.qbss.scount"], label="BYU-WiFi Station Count")

    
    
    # Plot total_adc
    #plt.plot(radio_df["aruba_erm.time"], radio_df["wlan.qbss.adc"] / 32767 * 100, label="ADC (%)")

    #plot adc% + cu
    #adc updates less frequently than cu, but seems to be well correlated
    #plt.plot(radio_df["aruba_erm.time"], 100 - ((radio_df["wlan.qbss.adc"] / 32767 * 100) + (radio_df["wlan.qbss.cu"] / 255 * 100)), label="ADC + CU")



    # Add labels and title
    plt.xlabel(f"{start_time}   to   {end_time}", fontsize=14)
    plt.ylabel("Values", fontsize=14)
    plt.title(f"{AP_name} Radio {plot_radio_number}", fontsize=16)

    # Set the y-axis limits from 0 to 100
    plt.ylim(0, 100)

    # Format x-axis to show only time (hour:minute)
    #showing GMT, this is broken
    # time_formatter = mdates.DateFormatter("%H:%M")  # Format: Hour:Minute
    # plt.gca().xaxis.set_major_formatter(time_formatter)

    # Rotate x-axis labels for better visibility
    #plt.xticks(rotation=45)

    # Add a legend
    plt.legend()

    # Display the plot
    plt.show()

In [4]:
# Function to perform the analysis for a given radio DataFrame
def analyze_radio_group(radio_df, radio_number):
    #print("analyze_radio_group")
    # Ensure the DataFrame is sorted by timestamp
    radio_df = radio_df.sort_values(by="aruba_erm.time").reset_index(drop=True)

    # Count the total number of data points
    total_data_points = len(radio_df)

    # Calculate the greatest value of wlan.qbss.cu
    greatest_cutil = ((radio_df["wlan.qbss.cu"].max() / 255) * 100)

    # Calculate the least value of wlan.qbss.cu
    least_cutil = ((radio_df["wlan.qbss.cu"].min() / 255) * 100)

    # Count the number of data points where wlan.qbss.cu is greater than 178
    greater_than_178_count = (radio_df["wlan.qbss.cu"] > 178).sum()

    # Calculate the percentage of data points where wlan.qbss.cu is greater than 178
    percentage_greater_than_178 = (greater_than_178_count / total_data_points) * 100

    # Median wlan.qbss.cu
    median_cutil = ((radio_df["wlan.qbss.cu"].median() / 255) * 100)

    # Calculate the greatest value of wlan.qbss.scount
    greatest_scount = radio_df["total_scount"].max()

    # Calculate the median value of wlan.qbss.scount
    median_scount = radio_df["total_scount"].median()

    # Calculate the least value of wlan.qbss.scount
    least_scount = radio_df["total_scount"].min()

    # Print the results
    print()
    print(f"--- Radio Number {radio_number} ---")
    print(f"Total beacons: {total_data_points}")
    print(f"Number of beacons with high channel utilization: {greater_than_178_count}")
    print(f"Percentage of beacons with high channel utilization: {percentage_greater_than_178:.2f}%")
    print(f"Highest channel utilization: {greatest_cutil:.2f}%")
    print(f"Median channel utilization: {median_cutil:.2f}%")
    print(f"Lowest channel utilization: {least_cutil:.2f}%")
    print(f"Highest station count: {greatest_scount:.0f}")
    print(f"Median station count: {median_scount:.0f}")
    print(f"Lowest station count: {least_scount:.0f}")

    # Create a boolean mask for wlan.qbss.cu > 178
    condition = radio_df["wlan.qbss.cu"] > 178

    # Assign group numbers for consecutive periods where condition is True
    radio_df["group"] = (condition != condition.shift()).cumsum() * condition

    # Filter groups where wlan.qbss.cu > 178
    valid_groups = radio_df[radio_df["group"] > 0]

    # Check if there are valid groups (i.e., groups where wlan.qbss.cu > 178)
    if valid_groups.empty:
        print(f"No records found where wlan.qbss.cu > 178 for radio_number {radio_number}.")
    else:
       # Find the start and end of each group
        group_details = valid_groups.groupby("group").agg(
            first_index=("aruba_erm.time", "idxmin"),
            last_index=("aruba_erm.time", "idxmax")
        )

        # Extract the first and last timestamps directly
        group_details["first_in_sequence_time"] = valid_groups.groupby("group")["aruba_erm.time"].min()
        group_details["end_time"] = valid_groups.groupby("group")["aruba_erm.time"].max()

        # Calculate the duration between the start and end times, and add 1 second 
        # Aruba updates beacons once per 10, which is also the rate that we have captured. 1.024 is the time the first beacon represents
        group_details["duration"] = (group_details["end_time"] - group_details["first_in_sequence_time"]).dt.total_seconds() + 1.024

        # Find the group with the longest duration
        longest_group = group_details["duration"].idxmax()
        longest_row = group_details.loc[longest_group]

        # Extract details of the longest duration
        longest_duration = longest_row["duration"]
        longest_start_time = longest_row["first_in_sequence_time"]
        longest_end_time = longest_row["end_time"]

        # Print results for the specific radio_number
        print(f"Longest consecutive duration of high channel use: {longest_duration:.2f} seconds")
        print(f"First beacon of sequence: {longest_start_time}")
        print(f"Last beacon of sequence: {longest_end_time}")

In [5]:
# Receives the information for each AP and class information
# Reads the file and processes the information for that time/AP
def graph(AP_name, start_time, end_time):
    #print("graph")
    # Your graph function implementation here

    print(f"AP name: {AP_name}")

    # Define the base directory for the files
    base_dir = "Data"
    file_name = AP_name + ".parquet"

    # Load the data for the AP file into a dataframe
    # this could be improved by loading the file once per AP instead of once per class
    df = pd.read_parquet(
        f"{base_dir}/{file_name}",
        engine="pyarrow",
    )

    # copy the AP df for manipulation
    filtered_df = df.copy()

    # time column
    filtered_df.reset_index(inplace=True) #reset index so that the timestamp is in a column instead

    # Fix timestamp consistency
    # Localize only if the timestamps are naive
    if filtered_df["aruba_erm.time"].dt.tz is None:
        filtered_df["aruba_erm.time"] = filtered_df["aruba_erm.time"].dt.tz_localize("UTC")

    # Convert to UTC-7 (this can always be done after localization)
    filtered_df["aruba_erm.time"] = filtered_df["aruba_erm.time"].dt.tz_convert("Etc/GMT+7")

    # Filter the DataFrame by the date/time range
    filtered_df = filtered_df[(filtered_df['aruba_erm.time'] >= start_time) & (filtered_df['aruba_erm.time'] <= end_time)]

    # Ensure the DataFrame is sorted by timestamp
    filtered_df = filtered_df.sort_values(by="aruba_erm.time").reset_index(drop=True)


    # Convert wlan.qbss.scount to integer, handling NaN values
    filtered_df["wlan.qbss.scount"] = filtered_df["wlan.qbss.scount"].fillna(0).astype(int)
    filtered_df["wlan.qbss.cu"] = filtered_df["wlan.qbss.cu"].fillna(0).astype(int)
    filtered_df["wlan.qbss.adc"] = filtered_df["wlan.qbss.adc"].fillna(0).astype(int)

    # create wlan.ta_int column
    # Function to convert hex MAC address string to integer
    def mac_to_int(mac):
        return int(mac.replace(":", ""), 16)

    # Apply the conversion to the wlan.ta column
    filtered_df["wlan.ta_int"] = filtered_df["wlan.ta"].apply(mac_to_int)
    

    # Apply the function to calculate total_scount
    filtered_df = calculate_total_scount(filtered_df)

    # Verify the result
    #print(filtered_df[["wlan.ta_int", "wlan.qbss.scount", "total_scount"]].head())
    #print(filtered_df.dtypes)
    

    # Check Radio and SSID count
    # Verify all values in wlan.vs.aruba.ap_name are the same
    if filtered_df["wlan.vs.aruba.ap_name"].nunique() == 1:
        wlan_ap_name_value = filtered_df["wlan.vs.aruba.ap_name"].iloc[0]
        #print(f"Data from {wlan_ap_name_value}")
    else:
        raise ValueError("Values in wlan.vs.aruba.ap_name are not all the same.")

    # Count unique transmit addresses and corresponding SSIDs
    # ensures that data is clean and shows multi-radio APs
    # Group by wlan.ta and collect the corresponding wlan.ssid values
    bssid_info = (
        filtered_df.groupby("wlan.ta")["wlan.ssid"]
        .unique()  # Get unique SSIDs for each wlan.ta
        .reset_index()
        .rename(columns={"wlan.ta": "transmit_address", "wlan.ssid": "SSIDs"})
    )

    # Count the number of unique BSSIDs
    unique_bssid_count = bssid_info["transmit_address"].nunique()


    # add a column for unique radio number
    # Create the 'radio_number' column
    filtered_df = filtered_df.sort_values("wlan.ta_int").reset_index(drop=True)

    # Initialize the 'radio_number' column
    radio_number = 1
    radio_numbers = [radio_number]

    # Assign radio numbers based on 'wlan.ta_int'
    for i in range(1, len(filtered_df)):
        # Check if the difference between consecutive wlan.ta_int values is greater than 1
        if abs(filtered_df.loc[i, "wlan.ta_int"] - filtered_df.loc[i - 1, "wlan.ta_int"]) > 1:
            radio_number += 1  # Increment radio number if the condition is met
        radio_numbers.append(radio_number)

    # Add the radio_number column to the DataFrame
    filtered_df["radio_number"] = radio_numbers

    # Ensure the DataFrame is sorted by timestamp
    filtered_df = filtered_df.sort_values(by="aruba_erm.time").reset_index(drop=True)

    # Display the updated DataFrame
    #print(filtered_df.head())

    #print number of radios
    print(f"Number or 5GHz radios: {filtered_df['radio_number'].max()}")

    # Output the BSSID results
    print(f"Number of BSSIDs: {unique_bssid_count}")
    print("List of addresses and their corresponding SSIDs:")
    print(bssid_info)


    # Filter the DataFrame for records where radio_number is 1
    filtered_radio_1_df = filtered_df[filtered_df["radio_number"] == 1]
    # Filter the data for "eduroam" and "BYU-WiFi" SSIDs
    eduroam_radio_1_df = filtered_df[(filtered_df["wlan.ssid"] == "eduroam") & (filtered_df["radio_number"] == 1)]
    byu_wifi_radio_1_df = filtered_df[(filtered_df["wlan.ssid"] == "BYU-WiFi") & (filtered_df["radio_number"] == 1)]

    #  Filter the DataFrame for records where radio_number is 2
    filtered_radio_2_df = filtered_df[filtered_df["radio_number"] == 2]

    # Run the analysis for radio_number 1 (using filtered_radio_1_df)
    if not filtered_radio_1_df.empty:
        analyze_radio_group(filtered_radio_1_df, 1)
        # Plot for the first radio
        plot_channel_stations(filtered_radio_1_df, eduroam_radio_1_df, byu_wifi_radio_1_df, AP_name, start_time, end_time)

    # Run the analysis for radio_number 2 (using filtered_radio_2_df) only if it's not empty
    if not filtered_radio_2_df.empty:
        analyze_radio_group(filtered_radio_2_df, 2)
        # Filter the data for "eduroam" and "BYU-WiFi" SSIDs
        eduroam_radio_2_df = filtered_df[(filtered_df["wlan.ssid"] == "eduroam") & (filtered_df["radio_number"] == 2)]
        byu_wifi_radio_2_df = filtered_df[(filtered_df["wlan.ssid"] == "BYU-WiFi") & (filtered_df["radio_number"] == 2)]
        # Plot for the second radio
        plot_channel_stations(filtered_radio_2_df, eduroam_radio_2_df, byu_wifi_radio_2_df, AP_name, start_time, end_time)
    else:
        print(f"{AP_name} has only one 5GHz radio")

    #print()



In [10]:
#read in sample files
# this can be done after pickle files have been created from samples.py
sample_classes_df = pd.read_pickle('sample_classes.pkl')
sample_aps_df = pd.read_pickle('sample_aps.pkl')
sample_classrooms_df = pd.read_pickle('sample_classrooms.pkl')

In [11]:
print(sample_classes_df.head())


    location    course                start_time                  end_time  \
0     EB-404    CE 529 2024-11-04 08:00:00-07:00 2024-11-04 08:50:00-07:00   
1  JFSB-B042  PHIL 110 2024-11-04 08:00:00-07:00 2024-11-04 09:15:00-07:00   
2  JFSB-B160   ASL 101 2024-11-04 08:00:00-07:00 2024-11-04 08:50:00-07:00   
3   JKB-1005  WRTG 313 2024-11-04 08:00:00-07:00 2024-11-04 08:50:00-07:00   
4   JKB-1006  SPAN 321 2024-11-04 08:00:00-07:00 2024-11-04 08:50:00-07:00   

   enrolled  capacity  
0        14        69  
1        31        44  
2        17        45  
3        23        30  
4        28        42  


In [12]:

print(sample_aps_df.head())


                 Hostname Building  Room  Number Building-Room  Model Group  \
0  P-BNSN-W003-AP-01-W131     BNSN  W003       1     BNSN-W003    515     A   
1    P-BRMB-380-AP-01-265     BRMB   380       1      BRMB-380    515     A   
2      P-CB-369-AP-01-322       CB   369       1        CB-369    515     A   
3      P-CB-392-AP-01-322       CB   392       1        CB-392    515     A   
4     P-CTB-385-AP-01-535      CTB   385       1       CTB-385    515     A   

  Notes  
0   NaN  
1   NaN  
2   NaN  
3   NaN  
4   NaN  


In [13]:
print(sample_classrooms_df)

     Location Building  Room  Capacity Group   Schedule  ap_count
0   BNSN-W003     BNSN  W003        25     A  BNSN-W003         1
1    BRMB-380     BRMB   380        48     A   BRMB-380         1
2      CB-369       CB   369        40     A     CB-369         1
3      CB-392       CB   392        43     A     CB-392         1
4     CTB-385      CTB   385        30     A    CTB-385         1
5      EB-404       EB   404        69     B     EB-404         1
6    ESC-C261      ESC  C261        48     A   ESC-C261         1
7    ESC-C396      ESC  C396        24     A   ESC-C396         1
8   HBLL-3710     HBLL  3710        57     B  HBLL-3710         1
9    HRCB-116     HRCB   116        30     A   HRCB-116         1
10  JFSB-B042     JFSB  B042        44     A  JFSB-B042         1
11  JFSB-B160     JFSB  B160        45     A  JFSB-B160         1
12  JFSB-B164     JFSB  B164        48     A  JFSB-B164         1
13   JKB-1005      JKB  1005        30     A   JKB-1005         1
14   JKB-1

In [ ]:
# run analysis for only one location
# Specify the target location
target_location = "CTB-385"

# Loop through each row in sample_classes_df
for index, class_row in sample_classes_df.iterrows():
    location = class_row['location']
    
    # Skip rows that don't match the target location
    if location != target_location:
        continue
    
    start_time = class_row['start_time']
    end_time = class_row['end_time']
    course = class_row['course']
    enrolled = class_row['enrolled']
    capacity = class_row['capacity']
    ap_count = sample_classrooms_df.loc[sample_classrooms_df['Location'] == location, 'ap_count'].iloc[0]


    print("\n")
    print(f"Subject: {course}")
    print(f"Location: {location}")
    print(f"Start Time: {start_time}")
    print(f"End Time: {end_time}")
    print(f"Capacity: {capacity}")
    print(f"Enrolled: {enrolled}")
    print(f"Number of APs: {ap_count}")
    print()
    
    # Sub loop for each matching AP in sample_aps_df
    for ap_index, ap_row in sample_aps_df.iterrows():
        building_room = ap_row['Building-Room']
        
        # Check if the building-room matches the location
        if building_room == location:
            AP_name = ap_row['Hostname']
            
            
            # Call the graph function with the variables
            graph(AP_name, start_time, end_time)


In [ ]:
# loop for each class row, iterate through each AP at that location
# this will go through ALL data and take a while
# graph function at top


# Clean up column names for consistency (strip spaces)
sample_classes_df.columns = sample_classes_df.columns.str.strip()
sample_aps_df.columns = sample_aps_df.columns.str.strip()

# Loop through each row in sample_classes_df
for index, class_row in sample_classes_df.iterrows():
    location = class_row['location']
    start_time = class_row['start_time']
    end_time = class_row['end_time']
    course = class_row['course']
    enrolled = class_row['enrolled']
    capacity = class_row['capacity']
    ap_count = sample_classrooms_df.loc[sample_classrooms_df['Location'] == location, 'ap_count'].iloc[0]


    print("\n")
    print(f"Subject: {course}")
    print(f"Location: {location}")
    print(f"Start Time: {start_time}")
    print(f"End Time: {end_time}")
    print(f"Capacity: {capacity}")
    print(f"Enrolled: {enrolled}")
    print(f"AP count {ap_count}")
    print()
    
    # Sub loop for each matching AP in sample_aps_df
    for ap_index, ap_row in sample_aps_df.iterrows():
        building_room = ap_row['Building-Room']
        
        # Check if the building-room matches the location
        if building_room == location:
            AP_name = ap_row['Hostname']
            
            # Call the graph function with the variables
            graph(AP_name, start_time, end_time)


In [317]:
#debug
# # Check all variables in the global scope and filter DataFrames
dataframes = {name: obj for name, obj in globals().items() if isinstance(obj, pd.DataFrame)}

# Print the names of the DataFrames
print("DataFrames in memory:", list(dataframes.keys()))

# Print the head of each DataFrame
for name, df in dataframes.items():
    print(f"DataFrame: {name}")
    print(df.head(), "\n")  # Print the first 5 rows


DataFrames in memory: ['df', 'filtered_df', 'valid_groups', 'group_durations', 'group_details', 'bssid_info', 'eduroam_df', 'byu_wifi_df', 'filtered_radio_1', 'eduroam_radio_1', 'byu_wifi_radio_1', 'filtered_radio_1_df', 'eduroam_radio_1_df', 'byu_wifi_radio_1_df', 'filtered_radio_2_df', 'eduroam_radio_2_df', 'byu_wifi_radio_2_df']
DataFrame: df
                                      wlan.ta wlan.vs.aruba.ap_name wlan.ssid  \
aruba_erm.time                                                                  
2024-11-03 01:00:00.140574  d0:4d:c6:df:a7:d1    P-EB-404-AP-01-419   eduroam   
2024-11-03 01:00:00.550216  d0:4d:c6:df:a7:e1    P-EB-404-AP-01-419   eduroam   
2024-11-03 01:00:00.959816  d0:4d:c6:df:a7:e0    P-EB-404-AP-01-419  BYU-WiFi   
2024-11-03 01:00:01.062176  d0:4d:c6:df:a7:d0    P-EB-404-AP-01-419  BYU-WiFi   
2024-11-03 01:00:01.164615  d0:4d:c6:df:a7:d1    P-EB-404-AP-01-419   eduroam   

                            wlan.qbss.scount  wlan.qbss.cu  wlan.qbss.adc  
aruba_er